# Real-data 3D voxel viewer: ligand · pocket · density

This notebook loads one real VoxBind/PDBbind sample and overlays its ligand, pocket, and electron density in the same 64³ voxel frame. Ligand and pocket atoms are shown as ball-and-stick models with optional opaque occupancy surfaces.

The default sample is `6ay5`, selected for its 129-heavy-atom pocket with no isolated atoms, fully contained single-component ligand, and complete density coverage at atomic sites. Change `SAMPLE_ID` below to inspect another sample with matching atom, density, and structure files.

## Setup

`voxels_ligvdw/atoms` stores seven ligand channels and four pocket channels. `voxels_v5/density` stores the normalized 2Fo–Fc density crop in the same ligand-centered frame. The deposited SDF/PDB coordinates are recentered using the identical all-heavy-atom ligand centroid.

For smooth browser rotation, atom layers use block-maximum reduction and density uses block-mean reduction to render at 16³. The original 64³ arrays remain available above.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display
from rdkit import Chem
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
from skimage.measure import find_contours

# ── User settings ───────────────────────────────────────────────────────────
SAMPLE_ID = "6ay5"
GRID_DIM = 64
RESOLUTION = 0.25  # Å / voxel
VIEW_STEP = 1      # full 64³ grid; preserves voxel-level density alignment


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "voxbind" / "dataset" / "data" / "pdbbind").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the VoxBind repository root.")


REPO_ROOT = find_repo_root()
PDBBIND_ROOT = REPO_ROOT / "voxbind" / "dataset" / "data" / "pdbbind"
ATOM_DIR = PDBBIND_ROOT / "voxels_ligvdw" / "atoms"
DENSITY_DIR = PDBBIND_ROOT / "voxels_v5" / "density"
STRUCTURE_ROOT = PDBBIND_ROOT / "structures" / "misato_qm_built"


def sample_paths(pid):
    pid = pid.lower()
    return ATOM_DIR / f"{pid}.npy", DENSITY_DIR / f"{pid}.npy"


def structure_paths(pid):
    pid = pid.lower()
    root = STRUCTURE_ROOT / pid
    return root / f"{pid}_ligand.sdf", root / f"{pid}_pocket.pdb"


def sample_is_complete(pid):
    atom_file, density_file = sample_paths(pid)
    ligand_file, pocket_file = structure_paths(pid)
    return all(path.exists() for path in (atom_file, density_file, ligand_file, pocket_file))


atom_path, density_path = sample_paths(SAMPLE_ID)
if not sample_is_complete(SAMPLE_ID):
    common = sorted(p.stem for p in DENSITY_DIR.glob("*.npy") if sample_is_complete(p.stem))
    if not common:
        raise FileNotFoundError("No complete atom/density/structure sample was found.")
    SAMPLE_ID = common[0]
    atom_path, density_path = sample_paths(SAMPLE_ID)
    print(f"Requested sample was unavailable; using {SAMPLE_ID!r}.")

print(f"repo       : {REPO_ROOT}")
print(f"sample     : {SAMPLE_ID}")
print(f"atom voxel : {atom_path.relative_to(REPO_ROOT)}")
print(f"density    : {density_path.relative_to(REPO_ROOT)}")

In [ ]:
atom_voxels = np.asarray(np.load(atom_path), dtype=np.float32)
density = np.asarray(np.load(density_path), dtype=np.float32)

if atom_voxels.shape != (11, GRID_DIM, GRID_DIM, GRID_DIM):
    raise ValueError(f"Expected atom voxels (11, 64, 64, 64), got {atom_voxels.shape}.")
if density.shape != (GRID_DIM, GRID_DIM, GRID_DIM):
    raise ValueError(f"Expected density (64, 64, 64), got {density.shape}.")

# Aggregate element channels only for display. The source tensor remains untouched.
ligand_voxels = atom_voxels[:7].max(axis=0)
pocket_voxels = atom_voxels[7:11].max(axis=0)


def read_sdf_ball_stick(path):
    lines = path.read_text().splitlines()
    n_atoms, n_bonds = int(lines[3][:3]), int(lines[3][3:6])
    coords = np.array([
        [float(line[0:10]), float(line[10:20]), float(line[20:30])]
        for line in lines[4:4 + n_atoms]
    ], dtype=np.float32)
    elements = np.array([line[31:34].strip() for line in lines[4:4 + n_atoms]])
    bonds = np.array([
        [int(line[0:3]) - 1, int(line[3:6]) - 1]
        for line in lines[4 + n_atoms:4 + n_atoms + n_bonds]
    ], dtype=np.int32)
    return coords, elements, bonds


def read_pdb_ball_stick(path):
    mol = Chem.MolFromPDBFile(str(path), sanitize=False, removeHs=False, proximityBonding=True)
    if mol is None:
        raise ValueError(f"RDKit could not parse {path}.")
    conformer = mol.GetConformer()
    coords = np.array([list(conformer.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())], dtype=np.float32)
    elements = np.array([atom.GetSymbol() for atom in mol.GetAtoms()])
    bonds = np.array([[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()] for bond in mol.GetBonds()], dtype=np.int32)
    return coords, elements, bonds


def heavy_atoms_only(coords, elements, bonds):
    keep = np.flatnonzero(~np.isin(np.char.upper(elements), ["H", "D", "T"]))
    remap = np.full(len(elements), -1, dtype=np.int32)
    remap[keep] = np.arange(len(keep))
    keep_bonds = np.all(np.isin(bonds, keep), axis=1) if len(bonds) else np.zeros(0, dtype=bool)
    return coords[keep], elements[keep], remap[bonds[keep_bonds]]


ligand_path, pocket_path = structure_paths(SAMPLE_ID)
ligand_coords_raw, ligand_elements_raw, ligand_bonds_raw = read_sdf_ball_stick(ligand_path)
ligand_center = ligand_coords_raw[~np.isin(np.char.upper(ligand_elements_raw), ["H", "D", "T"])].mean(axis=0)
ligand_coords, ligand_elements, ligand_bonds = heavy_atoms_only(
    ligand_coords_raw - ligand_center, ligand_elements_raw, ligand_bonds_raw
)
pocket_coords_raw, pocket_elements_raw, pocket_bonds_raw = read_pdb_ball_stick(pocket_path)
pocket_coords, pocket_elements, pocket_bonds = heavy_atoms_only(
    pocket_coords_raw - ligand_center, pocket_elements_raw, pocket_bonds_raw
)

ligand_atoms = len(ligand_coords)
pocket_atoms = len(pocket_coords)
density_stats = np.percentile(density, [1, 50, 95, 99, 99.9])

display(Markdown(
    f"**{SAMPLE_ID.upper()}** · atom tensor `{atom_voxels.shape}` · density `{density.shape}`  \\n"
    f"ligand heavy atoms: `{ligand_atoms}` · pocket heavy atoms: `{pocket_atoms}`  \\n"
    f"density percentiles (1/50/95/99/99.9%): `{np.round(density_stats, 3).tolist()}`"
))

In [ ]:
def block_reduce(volume, step=2, reducer="mean"):
    if step == 1:
        return volume.copy()
    n = (volume.shape[0] // step) * step
    trimmed = volume[:n, :n, :n]
    blocks = trimmed.reshape(n // step, step, n // step, step, n // step, step)
    axes = (1, 3, 5)
    if reducer == "max":
        return blocks.max(axis=axes)
    if reducer == "mean":
        return blocks.mean(axis=axes)
    raise ValueError(f"Unknown reducer: {reducer}")


ligand_view = block_reduce(ligand_voxels, VIEW_STEP, "max")
pocket_view = block_reduce(pocket_voxels, VIEW_STEP, "max")
density_view = block_reduce(density, VIEW_STEP, "mean")

n_view = ligand_view.shape[0]
centers = (
    np.arange(n_view, dtype=np.float32) * VIEW_STEP
    + (VIEW_STEP - 1) / 2
    - GRID_DIM / 2
) * RESOLUTION
X, Y, Z = np.meshgrid(centers, centers, centers, indexing="ij")

print(f"display grid: {ligand_view.shape} · physical crop: {GRID_DIM * RESOLUTION:.1f} Å per side")

## Interactive 3D view

The default view shows molecular geometry, contour-cage density, and the voxel lattice without in-view labels. Density iso defaults to `0.1`; contours are extracted independently from regularly spaced x/y/z voxel planes and overlaid in 3D with opacity fixed at `1.0`. A lightly smoothed display copy, short-contour filtering, and a soft proximity window keep density fully visible through 2.5 Å from ligand/pocket atoms before fading it out over the next 0.5 Å. Use the controls around the 3D scene to toggle layers and adjust density iso; drag to orbit and scroll to zoom.

In [ ]:
LIGAND_COLOR = "#F5B27E"
POCKET_COLOR = "#8291E8"
LIGAND_OCCUPANCY_COLOR = "#F8D3B0"
POCKET_OCCUPANCY_COLOR = "#B4BEF0"
PRIMARY_ATOM_OCCUPANCY_OPACITY = 0.4
DENSITY_COLOR = "#B58FDB"
PRIMARY_DENSITY_ISO = 0.1
DENSITY_CONTOUR_STRIDE = 2  # draw contours every two voxel planes
DENSITY_CONTOUR_WIDTH = 2.0
DENSITY_DISPLAY_SMOOTH_SIGMA = 0.75  # voxels; display only, source density is untouched
DENSITY_MIN_CONTOUR_POINTS = 12      # discard tiny fragmented contour loops
DENSITY_PROXIMITY_FULL_RADIUS_A = 2.5  # preserve density fully through 2.5 Å
DENSITY_PROXIMITY_FEATHER_A = 0.5      # fade from 2.5 to 3.0 Å
PRIMARY_DENSITY_OPACITY = 1.0
MASKED_DENSITY_OPACITY = 1.0
GRID_COLOR = "#323232"
GRID_LINE_WIDTH = 0.36  # half the previous width of 0.72
ATOM_COLORS = {
    "C": "#8c8c8c", "N": "#3050f8", "O": "#ff0d0d",
    "S": "#e0b000", "P": "#ff8000", "F": "#60d060",
    "Cl": "#1fa51f", "Br": "#a62929", "I": "#940094",
}
LIGAND_CARBON = LIGAND_COLOR
POCKET_CARBON = POCKET_COLOR
BALL_RADIUS_A = 0.32
STICK_RADIUS_A = BALL_RADIUS_A  # equal ball/stick radii, as used in the overview figure
SPHERE_LATITUDES = 8
SPHERE_LONGITUDES = 12
CYLINDER_SIDES = 8
GRID_SPACING_A = 4.0
FIGURE_SIZE = 760
HALF_EXTENT = GRID_DIM * RESOLUTION / 2


def flat_surface(
    volume, threshold, color, name, legendgroup, visible=True, opacity=1.0, surface_fill=1.0,
):
    return go.Isosurface(
        x=X.ravel(), y=Y.ravel(), z=Z.ravel(), value=volume.ravel(),
        isomin=float(threshold), isomax=float(threshold) + 0.01,
        surface=dict(count=1, fill=float(surface_fill), pattern="all"),
        colorscale=[[0.0, color], [1.0, color]], showscale=False,
        opacity=opacity, name=name, legendgroup=legendgroup,
        visible=visible if visible else "legendonly",
        caps=dict(x_show=False, y_show=False, z_show=False), hoverinfo="skip",
    )


def contour_line_coordinates(volume, threshold, stride=DENSITY_CONTOUR_STRIDE):
    line_x, line_y, line_z = [], [], []
    grid_indices = np.arange(volume.shape[0], dtype=np.float32)
    for fixed_axis in range(3):
        free_axes = [axis for axis in range(3) if axis != fixed_axis]
        for fixed_index in range(0, volume.shape[fixed_axis], stride):
            plane = np.take(volume, fixed_index, axis=fixed_axis)
            for contour in find_contours(plane, level=float(threshold)):
                if len(contour) < DENSITY_MIN_CONTOUR_POINTS:
                    continue
                xyz = np.empty((len(contour), 3), dtype=np.float32)
                xyz[:, fixed_axis] = centers[fixed_index]
                xyz[:, free_axes[0]] = np.interp(contour[:, 0], grid_indices, centers)
                xyz[:, free_axes[1]] = np.interp(contour[:, 1], grid_indices, centers)
                line_x.extend([*xyz[:, 0], None])
                line_y.extend([*xyz[:, 1], None])
                line_z.extend([*xyz[:, 2], None])
    return line_x, line_y, line_z


def contour_trace(
    volume, threshold, color, name, legendgroup, width=DENSITY_CONTOUR_WIDTH, opacity=1.0,
):
    line_x, line_y, line_z = contour_line_coordinates(volume, threshold)
    return go.Scatter3d(
        x=line_x, y=line_y, z=line_z, mode="lines",
        line=dict(color=color, width=width), opacity=opacity,
        name=name, legendgroup=legendgroup, hoverinfo="skip", showlegend=False,
    )



def crop_ball_stick(coords, elements, bonds, extent):
    inside = np.all(np.abs(coords) <= extent + 0.15, axis=1)
    keep = np.flatnonzero(inside)
    remap = np.full(len(coords), -1, dtype=np.int32)
    remap[keep] = np.arange(len(keep))
    keep_bonds = np.all(inside[bonds], axis=1) if len(bonds) else np.zeros(0, dtype=bool)
    return coords[keep], elements[keep], remap[bonds[keep_bonds]]


def mesh_trace(vertices, faces, vertex_colors, name, legendgroup, opacity=1.0):
    vertices = np.asarray(vertices, dtype=np.float32).reshape(-1, 3)
    faces = np.asarray(faces, dtype=np.int32).reshape(-1, 3)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], vertexcolor=vertex_colors,
        name=name, legendgroup=legendgroup, showlegend=False, hoverinfo="skip",
        opacity=float(opacity),
        flatshading=False,
        # High ambient light prevents intersections from becoming dark bands.
        lighting=dict(ambient=0.88, diffuse=0.35, specular=0.10, roughness=0.88, fresnel=0.01),
        lightposition=dict(x=100, y=180, z=260),
    )


def atom_color(element, model_color):
    # Deliberately hide element identity: one uniform color per molecular role.
    return model_color


def append_uv_sphere(vertices, faces, colors, center, radius, color):
    # Use one vertex at each pole. Repeated pole vertices create degenerate
    # triangles, which show up as dark wedges with smooth WebGL lighting.
    north = len(vertices)
    vertices.append(center + np.array([0.0, 0.0, radius]))
    colors.append(color)

    first_ring = len(vertices)
    for latitude in range(1, SPHERE_LATITUDES):
        theta = np.pi * latitude / SPHERE_LATITUDES
        for longitude in range(SPHERE_LONGITUDES):
            phi = 2 * np.pi * longitude / SPHERE_LONGITUDES
            direction = np.array([
                np.sin(theta) * np.cos(phi),
                np.sin(theta) * np.sin(phi),
                np.cos(theta),
            ])
            vertices.append(center + radius * direction)
            colors.append(color)

    south = len(vertices)
    vertices.append(center + np.array([0.0, 0.0, -radius]))
    colors.append(color)

    for longitude in range(SPHERE_LONGITUDES):
        next_longitude = (longitude + 1) % SPHERE_LONGITUDES
        faces.append((north, first_ring + longitude, first_ring + next_longitude))

    for ring in range(SPHERE_LATITUDES - 2):
        current = first_ring + ring * SPHERE_LONGITUDES
        following = current + SPHERE_LONGITUDES
        for longitude in range(SPHERE_LONGITUDES):
            next_longitude = (longitude + 1) % SPHERE_LONGITUDES
            a, b = current + longitude, current + next_longitude
            c, d = following + longitude, following + next_longitude
            faces.extend([(a, c, b), (b, c, d)])

    last_ring = first_ring + (SPHERE_LATITUDES - 2) * SPHERE_LONGITUDES
    for longitude in range(SPHERE_LONGITUDES):
        next_longitude = (longitude + 1) % SPHERE_LONGITUDES
        faces.append((last_ring + longitude, south, last_ring + next_longitude))


def atom_trace(coords, elements, name, legendgroup, carbon_color, radius=BALL_RADIUS_A):
    vertices, faces, colors = [], [], []
    for center, element in zip(coords, elements):
        append_uv_sphere(vertices, faces, colors, center, radius, atom_color(element, carbon_color))
    return mesh_trace(vertices, faces, colors, name, legendgroup)


def append_cylinder(vertices, faces, colors, start, end, radius, color):
    axis = np.asarray(end) - np.asarray(start)
    length = np.linalg.norm(axis)
    if length < 1e-6:
        return
    axis /= length
    helper = np.array([1.0, 0.0, 0.0]) if abs(axis[0]) < 0.8 else np.array([0.0, 1.0, 0.0])
    u = np.cross(axis, helper)
    u /= np.linalg.norm(u)
    v = np.cross(axis, u)
    offset = len(vertices)
    for point in (start, end):
        for side in range(CYLINDER_SIDES):
            angle = 2 * np.pi * side / CYLINDER_SIDES
            vertices.append(point + radius * (np.cos(angle) * u + np.sin(angle) * v))
            colors.append(color)
    for side in range(CYLINDER_SIDES):
        next_side = (side + 1) % CYLINDER_SIDES
        a, b = offset + side, offset + next_side
        c, d = offset + CYLINDER_SIDES + side, offset + CYLINDER_SIDES + next_side
        # Open-ended cylinder: atom spheres cover the ends, so caps would only
        # create coincident faces and dark z-fighting rings.
        faces.extend([(a, c, b), (b, c, d)])


def bond_trace(coords, elements, bonds, name, legendgroup, carbon_color, radius=STICK_RADIUS_A):
    vertices, faces, colors = [], [], []
    for start_index, end_index in bonds:
        start, end = coords[start_index], coords[end_index]
        # One continuous cylinder per bond; splitting at the midpoint produced
        # overlapping caps and the regular dark bands seen in screenshots.
        append_cylinder(vertices, faces, colors, start, end, radius, carbon_color)
    return mesh_trace(vertices, faces, colors, name, legendgroup)


def full_voxel_grid(extent, spacing=GRID_SPACING_A, visible=True):
    # Draw the lattice through the complete crop, not only on the outer faces.
    values = np.arange(-extent, extent + 1e-6, spacing)
    xs, ys, zs = [], [], []

    def add_line(a, b):
        xs.extend([a[0], b[0], None])
        ys.extend([a[1], b[1], None])
        zs.extend([a[2], b[2], None])

    for first in values:
        for second in values:
            add_line((-extent, first, second), (extent, first, second))
            add_line((first, -extent, second), (first, extent, second))
            add_line((first, second, -extent), (first, second, extent))

    return go.Scatter3d(
        x=xs, y=ys, z=zs, mode="lines",
        # name=f"Full voxel lattice ({spacing:g} Å)",
        line=dict(color=GRID_COLOR, width=GRID_LINE_WIDTH), hoverinfo="skip", visible=visible,
    )


density_smoothed_view = gaussian_filter(
    density_view, sigma=DENSITY_DISPLAY_SMOOTH_SIGMA, mode="nearest",
).astype(np.float32)
density_reference_coords = np.vstack((ligand_coords, pocket_coords))
density_atom_distance_view = cKDTree(density_reference_coords).query(
    np.column_stack((X.ravel(), Y.ravel(), Z.ravel())),
)[0].reshape(density_view.shape)
density_proximity_weight = np.clip(
    (DENSITY_PROXIMITY_FULL_RADIUS_A + DENSITY_PROXIMITY_FEATHER_A
     - density_atom_distance_view)
    / DENSITY_PROXIMITY_FEATHER_A,
    0.0, 1.0,
).astype(np.float32)
density_render_view = density_smoothed_view * density_proximity_weight
density_ceiling = float(max(1.0, np.percentile(density_render_view, 99.9)))

lig_coords_view, lig_elements_view, lig_bonds_view = crop_ball_stick(
    ligand_coords, ligand_elements, ligand_bonds, HALF_EXTENT
)
poc_coords_view, poc_elements_view, poc_bonds_view = crop_ball_stick(
    pocket_coords, pocket_elements, pocket_bonds, HALF_EXTENT
)


default_eye_direction = (-1.291620317953697, -1.291620317953697, 0.9783983126416558)
primary_density_view = density_render_view

fig = go.Figure([
    flat_surface(
        ligand_view, 0.45, LIGAND_OCCUPANCY_COLOR, "Ligand occupancy", "ligand-occupancy",
        visible=False, opacity=PRIMARY_ATOM_OCCUPANCY_OPACITY,
    ),
    flat_surface(
        pocket_view, 0.45, POCKET_OCCUPANCY_COLOR, "Pocket occupancy", "pocket-occupancy",
        visible=False, opacity=PRIMARY_ATOM_OCCUPANCY_OPACITY,
    ),
    contour_trace(
        primary_density_view, PRIMARY_DENSITY_ISO, DENSITY_COLOR,
        "Electron density ρ", "density", opacity=PRIMARY_DENSITY_OPACITY,
    ),
    full_voxel_grid(HALF_EXTENT, visible=True),
    bond_trace(poc_coords_view, poc_elements_view, poc_bonds_view, "Pocket sticks", "pocket-model", POCKET_CARBON),
    atom_trace(poc_coords_view, poc_elements_view, "Pocket balls/sticks", "pocket-model", POCKET_CARBON),
    bond_trace(lig_coords_view, lig_elements_view, lig_bonds_view, "Ligand sticks", "ligand-model", LIGAND_CARBON),
    atom_trace(lig_coords_view, lig_elements_view, "Ligand balls/sticks", "ligand-model", LIGAND_CARBON),
])
fig.data[3].name = "Voxel grid"
fig.data[5].showlegend = True
fig.data[7].showlegend = True

axis = dict(
    range=[-HALF_EXTENT, HALF_EXTENT],
    visible=False, showticklabels=False, ticks="", title="",
    showgrid=False, zeroline=False, showline=False, showbackground=False,
)
DEFAULT_CAMERA = dict(
    # x/y symmetry gives an exact 45° azimuth; up is the projected world-z
    # vector, exactly orthogonal to the view direction, so camera roll is zero.
    eye=dict(x=-1.291620317953697, y=-1.291620317953697, z=0.9783983126416558),
    center=dict(x=0.024726522758398496, y=0.024726522758398496, z=-0.127022385132688),
    up=dict(x=0.36102916038385824, y=0.36102916038385824, z=0.8598348043113008),
    projection=dict(type="perspective"),
)

fig.update_layout(
    width=FIGURE_SIZE, height=FIGURE_SIZE,
    margin=dict(l=0, r=0, t=0, b=0),
    showlegend=False, hovermode=False,
    scene=dict(
        xaxis=axis, yaxis=axis, zaxis=axis,
        aspectmode="cube",
        camera=DEFAULT_CAMERA, dragmode="orbit",
        bgcolor="white",
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    uirevision="camera-xy-neg45-level",
)

# Keep controls outside the 760 × 760 rendering square, as in the original visualizer.
primary_visualizer = go.FigureWidget(fig)
pocket_model_toggle = widgets.ToggleButton(
    value=True, description="Pocket ball-and-stick", icon="check",
    layout=widgets.Layout(width="150px"),
)
ligand_model_toggle = widgets.ToggleButton(
    value=True, description="Ligand ball-and-stick", icon="check",
    layout=widgets.Layout(width="150px"),
)
pocket_occupancy_toggle = widgets.ToggleButton(
    value=False, description="Pocket atom occupancy", icon="",
    layout=widgets.Layout(width="160px"),
)
ligand_occupancy_toggle = widgets.ToggleButton(
    value=False, description="Ligand atom occupancy", icon="",
    layout=widgets.Layout(width="160px"),
)
density_toggle = widgets.ToggleButton(
    value=True, description="Density", icon="check",
    layout=widgets.Layout(width="100px"),
)
density_iso_control = widgets.FloatSlider(
    value=PRIMARY_DENSITY_ISO, min=0.05, max=density_ceiling, step=0.05,
    description="Density iso:", readout_format=".2f", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="370px"),
)
def update_primary_layers(_=None):
    with primary_visualizer.batch_update():
        for trace_index in (4, 5):
            primary_visualizer.data[trace_index].visible = pocket_model_toggle.value
        for trace_index in (6, 7):
            primary_visualizer.data[trace_index].visible = ligand_model_toggle.value
        primary_visualizer.data[0].visible = ligand_occupancy_toggle.value
        primary_visualizer.data[1].visible = pocket_occupancy_toggle.value
        primary_visualizer.data[2].visible = density_toggle.value
    pocket_model_toggle.icon = "check" if pocket_model_toggle.value else ""
    ligand_model_toggle.icon = "check" if ligand_model_toggle.value else ""
    pocket_occupancy_toggle.icon = "check" if pocket_occupancy_toggle.value else ""
    ligand_occupancy_toggle.icon = "check" if ligand_occupancy_toggle.value else ""
    density_toggle.icon = "check" if density_toggle.value else ""


def update_primary_density(_=None):
    line_x, line_y, line_z = contour_line_coordinates(
        primary_density_view, density_iso_control.value,
    )
    with primary_visualizer.batch_update():
        primary_visualizer.data[2].x = line_x
        primary_visualizer.data[2].y = line_y
        primary_visualizer.data[2].z = line_z


pocket_model_toggle.observe(update_primary_layers, names="value")
ligand_model_toggle.observe(update_primary_layers, names="value")
pocket_occupancy_toggle.observe(update_primary_layers, names="value")
ligand_occupancy_toggle.observe(update_primary_layers, names="value")
density_toggle.observe(update_primary_layers, names="value")
density_iso_control.observe(update_primary_density, names="value")
layer_controls = widgets.HBox([
    pocket_model_toggle, ligand_model_toggle,
    pocket_occupancy_toggle, ligand_occupancy_toggle, density_toggle,
])
density_controls = widgets.HBox([density_iso_control])
display(widgets.VBox(
    [layer_controls, primary_visualizer, density_controls],
    layout=widgets.Layout(width=f"{FIGURE_SIZE}px"),
))

## Density-gradient-only view

Show the electron-density gradient magnitude `|∇ρ|` as a filled iso-surface with the aligned voxel lattice. The default iso threshold is `0.35`, and the view starts from the shared default camera.

In [ ]:
DENSITY_GRADIENT_COLOR = "#D9B7E4"  # keep the current gradmag color
GRADIENT_ISO = 0.35

gradient_x, gradient_y, gradient_z = np.gradient(density, RESOLUTION, edge_order=2)
density_gradient = np.sqrt(gradient_x ** 2 + gradient_y ** 2 + gradient_z ** 2).astype(np.float32)
density_gradient_view = block_reduce(density_gradient, VIEW_STEP, "max")
gradient_iso = GRADIENT_ISO
gradient_camera = DEFAULT_CAMERA

gradient_fig = go.Figure([
    flat_surface(
        density_gradient_view, gradient_iso, DENSITY_GRADIENT_COLOR,
        "Density gradient |∇ρ|", "density-gradient", visible=True,
        opacity=1.0, surface_fill=1.0,
    ),
    full_voxel_grid(HALF_EXTENT, visible=True),
])
gradient_fig.data[1].name = "Voxel grid"
gradient_fig.update_layout(
    width=FIGURE_SIZE, height=FIGURE_SIZE, margin=dict(l=0, r=0, t=0, b=0),
    showlegend=False, hovermode=False,
    scene=dict(
        xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube",
        camera=gradient_camera, dragmode="orbit", bgcolor="white",
    ),
    paper_bgcolor="white", plot_bgcolor="white",
    uirevision="camera-xy-neg45-level",
)

display(Markdown(f"Density-gradient iso: `{gradient_iso:.2f}`"))
display(gradient_fig)

## Random 3D patch masking

Apply a separated spatial mask (`8³` voxels per patch, `10%` mask ratio) to the three voxel modalities above. Mask patches are kept at least one block apart and prioritize regions intersecting ligand and pocket occupancy. The view uses the same basic rendering as the first figure: filled ligand/pocket occupancy surfaces followed by contour-line density. Masked informative patches are marked by cubes whose three camera-facing sides are filled with `#E9ECEC`; their fill and outline can be toggled together. The density iso threshold is `0.1`. The view starts from the shared default camera. Change `MASK_SEED` and rerun for a new reproducible mask.

In [ ]:
MASK_RATIO = 0.1
MASK_BLOCK_SIZE = 8       # voxels per side; 8 voxels = 2 Å
MASK_SEED = 7             # change this and rerun for another random mask
MASK_MIN_LIGAND_PATCHES = 3
MASK_MIN_POCKET_PATCHES = 6
SHOW_EMPTY_MASK_BLOCKS = False
MASK_FILL_COLOR = "#E9ECEC"
MASK_FILL_OPACITY = 0.8
MASK_LINE_COLOR = "#626262"
MASK_LINE_WIDTH = 4.8
MASK_OUTLINE_OFFSET_A = 0.01  # avoids z-fighting with filled faces
# Preserve each modality hue/lightness while targeting ~60% HSL saturation.
MASKED_LIGAND_COLOR = LIGAND_OCCUPANCY_COLOR
MASKED_POCKET_COLOR = POCKET_OCCUPANCY_COLOR
MASKED_LIGAND_OPACITY = 1.0
MASKED_POCKET_OPACITY = 1.0
MASKED_DENSITY_COLOR = DENSITY_COLOR
MASKED_DENSITY_ISO = 0.1
MASKED_DENSITY_CONTOUR_WIDTH = DENSITY_CONTOUR_WIDTH
MASK_GAP_A = 0.10         # small separation keeps neighboring cells visually distinct

if GRID_DIM % MASK_BLOCK_SIZE:
    raise ValueError("GRID_DIM must be divisible by MASK_BLOCK_SIZE.")
MASK_GRID_DIM = GRID_DIM // MASK_BLOCK_SIZE
MASK_BLOCK_WIDTH_A = MASK_BLOCK_SIZE * RESOLUTION


def informative_patch_mask(volume, threshold):
    # A patch is informative when it intersects the corresponding rendered iso-surface.
    blocks = volume.reshape(
        MASK_GRID_DIM, MASK_BLOCK_SIZE, MASK_GRID_DIM, MASK_BLOCK_SIZE, MASK_GRID_DIM, MASK_BLOCK_SIZE
    )
    return blocks.max(axis=(1, 3, 5)) >= threshold


def sample_separated_patch_mask(ratio, seed, prioritized_masks):
    target_count = int(round(ratio * MASK_GRID_DIM ** 3))
    rng = np.random.default_rng(seed)
    chosen = []

    def add_candidates(candidate_mask, limit=None):
        added = 0
        candidates = np.argwhere(candidate_mask)
        candidates = candidates[rng.permutation(len(candidates))]
        for candidate in candidates:
            if len(chosen) >= target_count or (limit is not None and added >= limit):
                break
            # Chebyshev distance > 1 prevents face-, edge-, and corner-touching cubes.
            if any(np.max(np.abs(candidate - previous)) <= 1 for previous in chosen):
                continue
            chosen.append(candidate.copy())
            added += 1

    for candidate_mask, limit in prioritized_masks:
        add_candidates(candidate_mask, limit)
    add_candidates(np.ones((MASK_GRID_DIM,) * 3, dtype=bool))
    if len(chosen) != target_count:
        raise RuntimeError(f"Could only place {len(chosen)}/{target_count} separated mask patches.")

    result = np.zeros((MASK_GRID_DIM,) * 3, dtype=bool)
    for block in chosen:
        result[tuple(block)] = True
    return result


def mask_box_traces(block_mask):
    fill_x, fill_y, fill_z = [], [], []
    fill_i, fill_j, fill_k = [], [], []
    outline_x, outline_y, outline_z = [], [], []
    cube_edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]

    def add_fill_face(low, high, fixed_axis, fixed_value, u_axis, v_axis):
        corners = []
        for u_high, v_high in ((False, False), (True, False), (True, True), (False, True)):
            point = low.copy()
            point[fixed_axis] = fixed_value
            point[u_axis] = high[u_axis] if u_high else low[u_axis]
            point[v_axis] = high[v_axis] if v_high else low[v_axis]
            corners.append(point)
        offset = len(fill_x)
        fill_x.extend(point[0] for point in corners)
        fill_y.extend(point[1] for point in corners)
        fill_z.extend(point[2] for point in corners)
        fill_i.extend([offset, offset])
        fill_j.extend([offset + 1, offset + 2])
        fill_k.extend([offset + 2, offset + 3])

    for block in np.argwhere(block_mask):
        low = -HALF_EXTENT + block * MASK_BLOCK_WIDTH_A + MASK_GAP_A / 2
        high = low + MASK_BLOCK_WIDTH_A - MASK_GAP_A
        outline_low = low - MASK_OUTLINE_OFFSET_A
        outline_high = high + MASK_OUTLINE_OFFSET_A
        outline_corners = [
            (outline_low[0], outline_low[1], outline_low[2]), (outline_high[0], outline_low[1], outline_low[2]),
            (outline_high[0], outline_high[1], outline_low[2]), (outline_low[0], outline_high[1], outline_low[2]),
            (outline_low[0], outline_low[1], outline_high[2]), (outline_high[0], outline_low[1], outline_high[2]),
            (outline_high[0], outline_high[1], outline_high[2]), (outline_low[0], outline_high[1], outline_high[2]),
        ]
        camera_side = np.where(np.asarray(default_eye_direction) >= 0, outline_high, outline_low)
        for start_index, end_index in cube_edges:
            start, end = outline_corners[start_index], outline_corners[end_index]
            on_camera_face = any(
                np.isclose(start[axis], camera_side[axis])
                and np.isclose(end[axis], camera_side[axis])
                for axis in range(3)
            )
            if not on_camera_face:
                continue
            outline_x.extend([start[0], end[0], None])
            outline_y.extend([start[1], end[1], None])
            outline_z.extend([start[2], end[2], None])
        for fixed_axis, u_axis, v_axis in ((0, 1, 2), (1, 0, 2), (2, 0, 1)):
            add_fill_face(low, high, fixed_axis, camera_side[fixed_axis], u_axis, v_axis)

    fill_trace = go.Mesh3d(
        x=fill_x, y=fill_y, z=fill_z, i=fill_i, j=fill_j, k=fill_k,
        color=MASK_FILL_COLOR, opacity=MASK_FILL_OPACITY, flatshading=True,
        lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
        name="Mask fill", hoverinfo="skip", showlegend=False,
    )
    outline_trace = go.Scatter3d(
        x=outline_x, y=outline_y, z=outline_z, mode="lines",
        line=dict(color=MASK_LINE_COLOR, width=MASK_LINE_WIDTH),
        name="Mask outline", hoverinfo="skip", showlegend=False,
    )
    return fill_trace, outline_trace


ligand_blocks = informative_patch_mask(ligand_voxels, 0.45)
pocket_blocks = informative_patch_mask(pocket_voxels, 0.45)
density_blocks = informative_patch_mask(density, MASKED_DENSITY_ISO)
informative_blocks = ligand_blocks | pocket_blocks | density_blocks
spatial_mask = sample_separated_patch_mask(
    MASK_RATIO, MASK_SEED,
    prioritized_masks=(
        (ligand_blocks, MASK_MIN_LIGAND_PATCHES),
        (pocket_blocks, MASK_MIN_POCKET_PATCHES),
        (informative_blocks, None),
    ),
)
display_mask = spatial_mask if SHOW_EMPTY_MASK_BLOCKS else spatial_mask & informative_blocks

if MASK_BLOCK_SIZE % VIEW_STEP:
    raise ValueError("MASK_BLOCK_SIZE must be divisible by VIEW_STEP.")
view_repeat = MASK_BLOCK_SIZE // VIEW_STEP
spatial_mask_view = spatial_mask.repeat(view_repeat, axis=0).repeat(view_repeat, axis=1).repeat(view_repeat, axis=2)
if spatial_mask_view.shape != ligand_view.shape:
    raise ValueError(f"Display mask {spatial_mask_view.shape} does not match volume {ligand_view.shape}.")

# Keep the arrays finite: Plotly can drop fragmented Isosurface traces when
# masked samples are NaN. Values below the iso thresholds remove the patch
# while preserving visible ligand, pocket, and density surfaces elsewhere.
masked_ligand_view = np.where(spatial_mask_view, 0.0, ligand_view)
masked_pocket_view = np.where(spatial_mask_view, 0.0, pocket_view)
masked_density_view = np.where(
    spatial_mask_view, MASKED_DENSITY_ISO - 1.0, density_render_view,
)

mask_fill_trace, mask_outline_trace = mask_box_traces(display_mask)
masked_fig = go.Figure([
    flat_surface(
        masked_pocket_view, 0.45, MASKED_POCKET_COLOR, "Pocket", "masked-pocket",
        visible=True, opacity=MASKED_POCKET_OPACITY,
    ),
    flat_surface(
        masked_ligand_view, 0.45, MASKED_LIGAND_COLOR, "Ligand", "masked-ligand",
        visible=True, opacity=MASKED_LIGAND_OPACITY,
    ),
    contour_trace(
        masked_density_view, MASKED_DENSITY_ISO, MASKED_DENSITY_COLOR,
        "Density ρ", "masked-density",
        width=MASKED_DENSITY_CONTOUR_WIDTH, opacity=MASKED_DENSITY_OPACITY,
    ),
    mask_fill_trace,
    mask_outline_trace,
    full_voxel_grid(HALF_EXTENT, visible=True),
])
masked_fig.data[-1].name = "Voxel grid"
actual_ratio = float(spatial_mask.mean())
masked_camera = DEFAULT_CAMERA
masked_fig.update_layout(
    width=FIGURE_SIZE, height=FIGURE_SIZE, margin=dict(l=0, r=0, t=0, b=0),
    showlegend=False, hovermode=False,
    scene=dict(
        xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube",
        camera=masked_camera, dragmode="orbit", bgcolor="white",
    ),
    paper_bgcolor="white", plot_bgcolor="white",
    uirevision=f"mask-{MASK_SEED}-camera-xy-neg45-level",
)

masked_visualizer = go.FigureWidget(masked_fig)
masked_ligand_toggle = widgets.ToggleButton(
    value=True, description="Ligand atom occupancy", icon="check",
    layout=widgets.Layout(width="180px"),
)
masked_pocket_toggle = widgets.ToggleButton(
    value=True, description="Pocket atom occupancy", icon="check",
    layout=widgets.Layout(width="180px"),
)
masked_blocks_toggle = widgets.ToggleButton(
    value=True, description="Masking blocks", icon="check",
    layout=widgets.Layout(width="150px"),
)


def update_masked_layers(_=None):
    with masked_visualizer.batch_update():
        masked_visualizer.data[0].visible = masked_pocket_toggle.value
        masked_visualizer.data[1].visible = masked_ligand_toggle.value
        masked_visualizer.data[3].visible = masked_blocks_toggle.value
        masked_visualizer.data[4].visible = masked_blocks_toggle.value
    masked_ligand_toggle.icon = "check" if masked_ligand_toggle.value else ""
    masked_pocket_toggle.icon = "check" if masked_pocket_toggle.value else ""
    masked_blocks_toggle.icon = "check" if masked_blocks_toggle.value else ""


masked_ligand_toggle.observe(update_masked_layers, names="value")
masked_pocket_toggle.observe(update_masked_layers, names="value")
masked_blocks_toggle.observe(update_masked_layers, names="value")
masked_layer_controls = widgets.HBox([
    masked_ligand_toggle, masked_pocket_toggle, masked_blocks_toggle,
])

display(Markdown(
    f"Masked spatial patches: `{spatial_mask.sum()}/{spatial_mask.size}` (`{actual_ratio:.1%}`) · "
    f"outlined informative patches shown: `{display_mask.sum()}` · "
    f"visible modalities: `ligand / pocket / density`"
))
display(widgets.VBox(
    [masked_layer_controls, masked_visualizer],
    layout=widgets.Layout(width=f"{FIGURE_SIZE}px"),
))

### Notes

- `ligand_voxels`, `pocket_voxels`, and `density` retain the original 64³ arrays.
- `VIEW_STEP = 1` renders the original 64³ voxel grid without block averaging, preserving voxel-level density alignment at the cost of slower loading.
- Change `SAMPLE_ID` to another PDB ID that has matching atom, density, SDF, and PDB files.
- Ball-and-stick colors are ligand `#F5B27E` and pocket `#8291E8`; occupancy colors are ligand `#F8D3B0` and pocket `#B4BEF0`. Density remains `#B58FDB`, and gradmag remains `#D9B7E4`.
- Primary and masked density both use iso `0.1` with 3D cross-sectional contour lines. Their line width is `2.0`; density opacity is fixed at `1.0`.
- Density contours use display-only Gaussian smoothing (`sigma=0.75`), omit loops shorter than 12 points, stay fully visible through 2.5 Å from ligand/pocket heavy atoms, and fade out from 2.5 to 3.0 Å; no camera-dependent sightline carving is applied.
- Gradmag uses a fully filled surface with iso `0.35`.
- Masked patches fill their three default-camera-facing sides with `#E9ECEC`; their nine bounding edges are outlined without hatching, and the three rear sides/edges are omitted. Outline color is `#626262` with line width `4.8`.
- First-view ligand/pocket atom occupancy uses opacity `0.4`. In the masked view, pocket is rendered before ligand and both are opaque (`1.0`) to avoid transparent WebGL depth-sorting artifacts; ligand, pocket, and masking blocks can be toggled independently.
- Every voxel grid uses color `#323232` and line width `0.36`.
- The voxel grid and density start visible; ligand and pocket occupancy surfaces can be enabled independently.
- Ligand and pocket ball-and-stick models use one uniform role color; atom types are intentionally hidden.
- Manual camera rotation and zoom remain available in every view.